# 05 · Exploratorio (NO pre-registrado): ¿por qué falla la sonda lineal H2-c?

**Advertencia.** Este análisis se agregó **después** de ver los resultados de H2 y no figura en el protocolo pre-registrado. Se reporta como exploración para diagnosticar un resultado inesperado, no como evidencia confirmatoria.

**Lo observado.** La sonda lineal 1×1 sobre el embedding del cuello (H2-c) obtuvo IoU 0.07 en valid, con precisión 0.68 y sensibilidad 0.08. La sonda por tokens (P, notebook 03) separa agua de tierra con AUC 0.92 en ese mismo cuello. Hay dos explicaciones posibles:

1. **Umbral mal calibrado.** Los logits ordenan bien los píxeles (AUC alta), pero el umbral 0 corta demasiado alto por el desbalance agua/tierra.
2. **Subajuste.** Con el mismo presupuesto que H2 (12 épocas, lr 1e-3), la sonda no llega a converger.

Se mide (a) el AUC a nivel de píxel, (b) el IoU con un umbral elegido **en train** (nunca en valid) y (c) una sonda entrenada más tiempo.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from inf659sam import config, data
from inf659sam.cache import load_embedding, save_json
from inf659sam.decoder_ft import load_training_set, train_probe

config.setup_runtime(seed=0)
parts = json.loads((config.RESULTS_DIR / "particiones_usadas.json").read_text(encoding="utf-8"))
TRAIN, VALID = parts["train"], parts["valid"]
items = load_training_set(TRAIN)
BINS = np.linspace(-20, 20, 801)


@torch.no_grad()
def logits_of(probe, chips):
    dev = next(probe.parameters()).device
    out = []
    for c in chips:
        lab = data.load_chip(c)["label"]
        lg = probe(load_embedding(c)["emb"].to(dev))[0].float().cpu().numpy()
        out.append((lg, lab))
    return out


def hist_counts(pairs):
    hw = np.zeros(len(BINS) + 1)
    hn = np.zeros(len(BINS) + 1)
    for lg, lab in pairs:
        v = lab != -1
        hw += np.bincount(np.digitize(lg[v & (lab == 1)], BINS), minlength=len(BINS) + 1)
        hn += np.bincount(np.digitize(lg[v & (lab == 0)], BINS), minlength=len(BINS) + 1)
    return hw, hn


def iou_at(hw, hn, k):
    """IoU global prediciendo agua para los bins >= k."""
    tp = hw[k:].sum()
    fp = hn[k:].sum()
    fn = hw[:k].sum()
    return tp / (tp + fp + fn)


def evaluar(probe, nombre):
    tr, va = logits_of(probe, TRAIN), logits_of(probe, VALID)
    hw_t, hn_t = hist_counts(tr)
    hw_v, hn_v = hist_counts(va)
    k0 = int(np.digitize(0.0, BINS))
    ks = np.arange(1, len(BINS))
    k_best = int(ks[np.argmax([iou_at(hw_t, hn_t, k) for k in ks])])
    rng = np.random.default_rng(0)
    ys, ss = [], []
    for lg, lab in va:
        v = np.flatnonzero(lab.ravel() != -1)
        pick = rng.choice(v, size=min(20000, len(v)), replace=False)
        ys.append((lab.ravel()[pick] == 1).astype(int))
        ss.append(lg.ravel()[pick])
    auc = roc_auc_score(np.concatenate(ys), np.concatenate(ss))
    return {"sonda": nombre, "AUC_pixel_valid": auc, "IoU_valid_umbral_0": iou_at(hw_v, hn_v, k0),
            "umbral_elegido_en_train": float(BINS[k_best - 1]), "IoU_valid_umbral_train": iou_at(hw_v, hn_v, k_best)}


filas = []
probe_a, h_a = train_probe(items, seed=0)  # idéntica a H2-c, semilla 0
filas.append(evaluar(probe_a, "H2-c original (12 épocas, lr 1e-3)") | {"perdida_final": h_a["historial"][-1]["perdida_media"]})
probe_b, h_b = train_probe(items, seed=0, lr=1e-2, epochs=60)
filas.append(evaluar(probe_b, "más presupuesto (60 épocas, lr 1e-2)") | {"perdida_final": h_b["historial"][-1]["perdida_media"]})
df = pd.DataFrame(filas)
save_json({"advertencia": "exploratorio, no pre-registrado", "filas": filas}, config.RESULTS_DIR / "exploratorio_sonda.json")
df.round(4)

Entorno: {'seed': 0, 'torch_threads': 8}


,sonda,AUC_pixel_valid,IoU_valid_umbral_0,umbral_elegido_en_train,IoU_valid_umbral_train,perdida_final
0,"H2-c original (12 épocas, lr 1e-3)",0.8307,0.0771,-1.45,0.2863,1.1134
1,"más presupuesto (60 épocas, lr 1e-2)",0.8696,0.3154,-1.45,0.3844,0.9709


**Cómo leerlo.** Si el AUC por píxel es alto y el IoU sube mucho al elegir el umbral en train, la información está en el embedding y el problema de H2-c es de **calibración del umbral**. Si además la sonda entrenada más tiempo mejora, había **subajuste**. En ambos casos, el decodificador ajustado (H2) resuelve las dos cosas a la vez: aprende a leer la señal y calibra su salida.